In [1]:
from pathlib import Path
import json
import os
import pandas as pd
from main import load_dotenv
from data_processing.music_representations.config import MusicRepresentationConfig
from storage.factory import build_storage

load_dotenv(Path("../.env"))

/home/skynet/research/MusicGeneration/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
output_dir = Path(os.environ.get("MUSIC_REPR_OUTPUT_DIR"))
config = MusicRepresentationConfig(
    input_dir=Path(os.environ.get("MUSIC_REPR_INPUT_DIR")),
    output_dir=output_dir,
)
storage = build_storage(config)

In [3]:
representation = "unigram"
representation_root = storage.materialize(representation)
segments_df = pd.read_parquet(representation_root / "segments.parquet")

In [4]:
train_df = segments_df[segments_df["maestro_split"] == "train"]

In [5]:
from torch.utils.data import Dataset, DataLoader

class MusicDataset(Dataset):
    def __init__(self, df, representation_root, load_fn):
        self.df = df.reset_index(drop=True)
        self.representation_root = Path(representation_root)
        self.load_fn = load_fn

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.load_fn(row, self.representation_root)

In [6]:
def load_music(row, representation_root):
    token_path = representation_root / row["token_file"]
    payload = json.loads(token_path.read_text(encoding="utf-8"))

    return {
        "piece_id": row["piece_id"],
        "segment_id": row["segment_id"],
        "tokens": payload["ids"],
    }

def raw_collate(samples):
    return samples

In [7]:
import torch
from torch.nn.utils.rnn import pad_sequence


def neural_collate(samples):
    sequences = [
        torch.tensor(sample["tokens"], dtype=torch.long)
        for sample in samples
    ]

    lengths = torch.tensor(
        [len(sequence) for sequence in sequences]
    )

    padded = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0,
    )

    return {
        "piece_id": [sample["piece_id"] for sample in samples],
        "segment_id": [sample["segment_id"] for sample in samples],
        "tokens": padded,
        "lengths": lengths,
    }

In [11]:
dataset = MusicDataset(train_df, representation_root, load_music)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=raw_collate,
)

In [12]:
representation_root, train_df.head()

(PosixPath('/home/skynet/research/data/maestro/maestro-v3.0.0_cononical/unigram'),
            piece_id segment_id maestro_split  \
 0  afd9c0e6bccdc705       None         train   
 4  6adadcbe38aa1bd4       None         train   
 5  cce535492f7d8fa5       None         train   
 6  91c584ff23f52837       None         train   
 7  c1046bb8348fa21d       None         train   
 
                           token_file  token_count  
 0  tokens/afd9c0e6bccdc705_full.json        30014  
 4  tokens/6adadcbe38aa1bd4_full.json        68179  
 5  tokens/cce535492f7d8fa5_full.json        21822  
 6  tokens/91c584ff23f52837_full.json        35678  
 7  tokens/c1046bb8348fa21d_full.json        12833  )

In [16]:
for batch in loader:
    print(len(batch))
    break

32
